# Huấn luyện mô hình nhận diện đồ nội thất

**Cách dùng:**

1. Bật GPU: menu **Runtime → Change runtime type → T4 GPU → Save**
2. Menu **Runtime → Run all**
3. Chờ khoảng 1,5 – 3 tiếng. **Để tab này mở**, không Colab tự ngắt.
4. Ô cuối cùng sẽ tự tải tệp `best.pt` về máy.
5. Chép `best.pt` vào thư mục `models` của phần mềm rồi chạy lại `chay.bat`.
   Phần mềm tự nhận ra và dùng, không phải sửa dòng mã nào.


## 1. Kiểm tra GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit('CHUA BAT GPU! Vao Runtime > Change runtime type > T4 GPU roi chay lai.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Cài thư viện (khoảng 2 phút)

In [ ]:
!pip install -q ultralytics fiftyone
print('Da cai xong')


## 3. Tải ảnh nội thất đã có sẵn nhãn

Lấy từ bộ **Open Images V7** của Google — ảnh đã được gán nhãn sẵn, không phải ngồi khoanh tay.

Muốn mô hình giỏi hơn thì tăng `SO_ANH_TRAIN`, đổi lại tải và huấn luyện lâu hơn.

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz

SO_ANH_TRAIN = 6000      # tang len 12000-15000 neu co thoi gian
SO_ANH_VAL   = 1200

LOP = ['Table', 'Desk', 'Coffee table', 'Kitchen & dining room table',
       'Chair', 'Stool', 'Bench', 'Couch', 'Bed',
       'Wardrobe', 'Cabinetry', 'Cupboard', 'Chest of drawers',
       'Nightstand', 'Drawer', 'Shelf', 'Bookcase', 'Filing cabinet']

train = foz.load_zoo_dataset('open-images-v7', split='train',
                             label_types=['detections'], classes=LOP,
                             max_samples=SO_ANH_TRAIN, dataset_name='nt_train')
val = foz.load_zoo_dataset('open-images-v7', split='validation',
                           label_types=['detections'], classes=LOP,
                           max_samples=SO_ANH_VAL, dataset_name='nt_val')

for bo, ten in ((train, 'train'), (val, 'val')):
    bo.export(export_dir='/content/noi_that',
              dataset_type=fo.types.YOLOv5Dataset,
              label_field='ground_truth', split=ten, classes=LOP)

print('Xong. Du lieu o /content/noi_that')


## 4. Huấn luyện (lâu nhất, khoảng 1–2 tiếng)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
ket_qua = model.train(
    data='/content/noi_that/dataset.yaml',
    epochs=60, imgsz=640, batch=16, patience=15,
    degrees=10, translate=0.1, scale=0.5, fliplr=0.5, mosaic=1.0,
    name='noi_that', project='/content/runs',
)
print('Huan luyen xong')


## 5. Đo độ chính xác

Chép bốn con số này vào báo cáo. **mAP@50 trên 0,70 là khá, trên 0,85 là tốt.**

In [ ]:
from ultralytics import YOLO
m = YOLO('/content/runs/noi_that/weights/best.pt')
kq = m.val(data='/content/noi_that/dataset.yaml')

print()
print('Precision   :', round(float(kq.box.mp), 4))
print('Recall      :', round(float(kq.box.mr), 4))
print('mAP@50      :', round(float(kq.box.map50), 4))
print('mAP@50-95   :', round(float(kq.box.map), 4))


## 6. Tải mô hình về máy

In [ ]:
from google.colab import files
files.download('/content/runs/noi_that/weights/best.pt')
print('Chep tep best.pt vua tai vao thu muc models cua phan mem, roi chay lai chay.bat')
